# Notebook 02: Sentence-BERT (SBERT) Multilingual Embeddings
**Project:** JobLens Vietnam - Labor Market Analysis & Skill Trends  
**Course:** Introduction to Data Science  

### Objective:
1. Explain the selection of `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`.
2. Demonstrate batch encoding for Vietnamese/English technical job descriptions.
3. Validate embedding dimensions (384-dimensional dense vectors) and unit normalization.
4. Align embeddings with tabular datasets using `unique_job_id` keys.
5. Save cached `.npy` arrays and metadata tracking file.

In [ ]:
import sys
from pathlib import Path
import json
import numpy as np
import pandas as pd

PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.config as config
from src.features.text_processor import prepare_classification_regression_text, prepare_clustering_text
from src.features.sbert_encoder import load_sbert_model, encode_texts, save_embedding_metadata

## 1. Load Pretrained SBERT Model
We utilize `sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`, an efficient 12-layer Transformer supporting 50+ languages including Vietnamese and English. It balances embedding quality with CPU inference efficiency.

In [ ]:
model = load_sbert_model()
print(f"Loaded SBERT model: {config.SBERT_MODEL_NAME}")
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")
print(f"Max sequence length: {model.max_seq_length}")

## 2. Load Processed Datasets and Text Representations
We extract:
1. `job_description` for Classification and Regression.
2. `job_title + job_description + requirements` for Unsupervised Clustering.

In [ ]:
cleaned_df = pd.read_csv(config.PROCESSED_DATA_DIR / "cleaned_jobs.csv")
print(f"Cleaned master records: {len(cleaned_df)}")

# Prepare text lists
desc_texts, desc_ids = prepare_classification_regression_text(cleaned_df)
print(f"Sample description text (first 150 chars):\n{desc_texts[0][:150]}...")

## 3. Verify or Compute Embeddings
Load cached embeddings from `embeddings/sbert_embeddings.npy` and `embeddings/sbert_clustering_embeddings.npy` if already generated.

In [ ]:
sbert_path = config.EMBEDDINGS_DIR / "sbert_embeddings.npy"
embeddings, _ = encode_texts(desc_texts, desc_ids, sbert_path, batch_size=config.SBERT_BATCH_SIZE)

print(f"SBERT Embeddings shape: {embeddings.shape}")
print(f"Vector dtype: {embeddings.dtype}")
print(f"Sample vector norm (should be ~1.0 if normalized): {np.linalg.norm(embeddings[0]):.4f}")

## 4. Embedding Metadata and Diagnostics
Inspect `embedding_metadata.json` which tracks dimensionality, sample identifiers, and row alignment.

In [ ]:
meta_path = config.EMBEDDINGS_DIR / "embedding_metadata.json"
if meta_path.exists():
    with open(meta_path, 'r', encoding='utf-8') as f:
        meta = json.load(f)
    print(json.dumps(meta, indent=2, ensure_ascii=False))
else:
    print("Metadata file will be created by the main pipeline.")